Preprocessing Dataset Film 

Notebook ini bertugas mempersiapkan katalog film kasar (`film_kasar.csv`) dari dataset IMDb Top 1000.  
Output akan digunakan pada tahap **Self-Labeling Sinopsis Film**.

**Langkah yang dilakukan:**
1. Setup variabel path
2. Load dataset mentah
3. Seleksi 3 kolom utama
4. Handling missing values pada kolom `Overview`
5. Simpan output ke `film_kasar.csv`

In [6]:
import os
import pandas as pd

In [7]:
#Setup Path Integrasi & Variabel PATH

BASE_DIR = os.path.abspath("../")
print(f"[INFO] BASE_DIR detected : {BASE_DIR}")

# Input path
CSV_PATH = os.path.join(BASE_DIR, "dataset_raw", "updated_imdb_top_1000.csv")
print(f"[INFO] CSV_PATH          : {CSV_PATH}")

# Output path
OUTPUT_DIR = os.path.join(BASE_DIR, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(OUTPUT_DIR, "film_kasar.csv")

print(f"[INFO] OUTPUT_PATH       : {OUTPUT_PATH}")

[INFO] BASE_DIR detected : c:\Users\VICTUS\OneDrive\Documents\Semester_4\NLP_LAB\FINPRO2
[INFO] CSV_PATH          : c:\Users\VICTUS\OneDrive\Documents\Semester_4\NLP_LAB\FINPRO2\dataset_raw\updated_imdb_top_1000.csv
[INFO] OUTPUT_PATH       : c:\Users\VICTUS\OneDrive\Documents\Semester_4\NLP_LAB\FINPRO2\data\film_kasar.csv


In [8]:
#Load Dataset Mentah
print("[INFO] Memuat dataset dari CSV_PATH ...")
df_raw = pd.read_csv(CSV_PATH)

print(f"[INFO] Dataset berhasil dimuat.")
print(f"Jumlah baris  : {len(df_raw):,}")
print(f"Jumlah kolom  : {len(df_raw.columns)}")
print(f"Daftar kolom  : {list(df_raw.columns)}")

[INFO] Memuat dataset dari CSV_PATH ...
[INFO] Dataset berhasil dimuat.
Jumlah baris  : 1,000
Jumlah kolom  : 25
Daftar kolom  : ['Series_Title', 'Released_Year', 'Certificate', 'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross', 'tmdb_id', 'tmdb_vote_average', 'tmdb_release_date', 'tmdb_original_language', 'tmdb_popularity', 'tmdb_genres', 'tmdb_budget', 'tmdb_revenue', 'tmdb_runtime', 'tmdb_status']


In [9]:
#Seleksi Fitur — Hanya 3 Kolom Utama
KOLOM_UTAMA = ['Series_Title', 'Genre', 'Overview']
df = df_raw[KOLOM_UTAMA].copy()

print(f"[INFO] Seleksi kolom selesai.")
print(f"Kolom dipertahankan : {KOLOM_UTAMA}")
print(f"Kolom dibuang       : {[c for c in df_raw.columns if c not in KOLOM_UTAMA]}")
print(f"Shape setelah seleksi : {df.shape}")

[INFO] Seleksi kolom selesai.
Kolom dipertahankan : ['Series_Title', 'Genre', 'Overview']
Kolom dibuang       : ['Released_Year', 'Certificate', 'Runtime', 'IMDB_Rating', 'Meta_score', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross', 'tmdb_id', 'tmdb_vote_average', 'tmdb_release_date', 'tmdb_original_language', 'tmdb_popularity', 'tmdb_genres', 'tmdb_budget', 'tmdb_revenue', 'tmdb_runtime', 'tmdb_status']
Shape setelah seleksi : (1000, 3)


In [10]:
print("[PREVIEW] 3 baris pertama:")
df.head(3)

[PREVIEW] 3 baris pertama:


,Series_Title,Genre,Overview
0,The Shawshank Redemption,Drama,Two imprisoned men bond over a number of years...
1,The Godfather,"Crime, Drama",An organized crime dynasty's aging patriarch t...
2,The Dark Knight,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...


In [11]:
#Handling Missing Values pada Kolom `Overview`
jumlah_baris_sebelum = len(df)
print(f"[LOG] Jumlah baris SEBELUM drop missing values : {jumlah_baris_sebelum:,}")


nan_count = df['Overview'].isna().sum()
print(f"[LOG] Baris dengan Overview = NaN            : {nan_count}")
empty_count = (df['Overview'].str.strip() == '').sum()
print(f"[LOG] Baris dengan Overview = string kosong  : {empty_count}")

df['Overview'] = df['Overview'].replace(r'^\s*$', pd.NA, regex=True)
df = df.dropna(subset=['Overview'])
df = df.reset_index(drop=True)

jumlah_baris_sesudah = len(df)
jumlah_drop = jumlah_baris_sebelum - jumlah_baris_sesudah


print(f"[LOG] Jumlah baris SESUDAH drop missing values : {jumlah_baris_sesudah:,}")
print(f"[LOG] Total baris yang di-drop                 : {jumlah_drop}")

if jumlah_drop == 0:
    print("[INFO] Tidak ada baris yang dihapus — semua sinopsis lengkap. Dataset siap diproses.")
else:
    print(f"[INFO] {jumlah_drop} baris berhasil dihapus karena Overview kosong/NaN.")

[LOG] Jumlah baris SEBELUM drop missing values : 1,000
[LOG] Baris dengan Overview = NaN            : 0
[LOG] Baris dengan Overview = string kosong  : 0
[LOG] Jumlah baris SESUDAH drop missing values : 1,000
[LOG] Total baris yang di-drop                 : 0
[INFO] Tidak ada baris yang dihapus — semua sinopsis lengkap. Dataset siap diproses.


In [12]:
#Verifikasi Akhir Dataset
print("RINGKASAN DATASET SETELAH PREPROCESSING")
print(f"Shape akhir          : {df.shape}")
print(f"Kolom                : {list(df.columns)}")
print("  Missing values per kolom:")
for col in df.columns:
    mv = df[col].isna().sum()
    print(f"- {col:<20}: {mv} missing")

RINGKASAN DATASET SETELAH PREPROCESSING
Shape akhir          : (1000, 3)
Kolom                : ['Series_Title', 'Genre', 'Overview']
  Missing values per kolom:
- Series_Title        : 0 missing
- Genre               : 0 missing
- Overview            : 0 missing


In [13]:
# Teks Overview TIDAK dibersihkan 
print("[VERIFIKASI] Contoh teks Overview (raw, tanpa pembersihan):")
for i, row in df.head(3).iterrows():
    print(f"  [{i}] {row['Series_Title']}")
    print(f"Genre   : {row['Genre']}")
    print(f"Overview: {row['Overview'][:120]}...")

[VERIFIKASI] Contoh teks Overview (raw, tanpa pembersihan):
  [0] The Shawshank Redemption
Genre   : Drama
Overview: Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency....
  [1] The Godfather
Genre   : Crime, Drama
Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son....
  [2] The Dark Knight
Genre   : Action, Crime, Drama
Overview: When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greates...


In [14]:
#Simpan Output ke `film_kasar.csv`
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[INFO] Folder OUTPUT tersedia di : {OUTPUT_DIR}")

# Simpan DataFrame ke CSV tanpa index
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f"[INFO] File berhasil disimpan   : {OUTPUT_PATH}")
print(f"[INFO] Jumlah baris tersimpan   : {len(df):,}")
print(f"[INFO] Jumlah kolom tersimpan   : {len(df.columns)}") 

[INFO] Folder OUTPUT tersedia di : c:\Users\VICTUS\OneDrive\Documents\Semester_4\NLP_LAB\FINPRO2\data
[INFO] File berhasil disimpan   : c:\Users\VICTUS\OneDrive\Documents\Semester_4\NLP_LAB\FINPRO2\data\film_kasar.csv
[INFO] Jumlah baris tersimpan   : 1,000
[INFO] Jumlah kolom tersimpan   : 3
